# Project Part 2 — Lossless Source Coding
## Text Source: Character-Level

In [1]:
import math, collections, heapq, os, subprocess
import matplotlib.pyplot as plt

with open('Text.txt', 'r', encoding='utf-8') as f:
    raw = f.read()
symbols = list(raw.lower())
n = len(symbols)

counts = collections.Counter(symbols)
probs  = {x: c/n for x,c in counts.items()}

def entropy(p): return -sum(v*math.log2(v) for v in p.values() if v>0)
H = entropy(probs)
print(f'Total symbols   : {n}')
print(f'Alphabet size   : {len(probs)}')
print(f'Entropy H(X)    : {H:.4f} bits/symbol')

Total symbols   : 201276
Alphabet size   : 49
Entropy H(X)    : 4.3104 bits/symbol


---
## Task 1 — Huffman Coding
### Q1a: Build Huffman tree from empirical probabilities

In [2]:
def build_huffman(prob_dict):
    # Each node: [probability, id, symbol_or_None, left, right]
    heap = []
    for i, (sym, p) in enumerate(prob_dict.items()):
        heapq.heappush(heap, [p, i, sym, None, None])
    node_id = len(prob_dict)
    while len(heap) > 1:
        l = heapq.heappop(heap)
        r = heapq.heappop(heap)
        merged = [l[0]+r[0], node_id, None, l, r]
        heapq.heappush(heap, merged)
        node_id += 1
    return heap[0]

def get_codes(tree, prefix='', codes={}):
    # Walk tree to extract codeword for each symbol
    if tree[2] is not None:  # leaf
        codes[tree[2]] = prefix if prefix else '0'
    else:
        get_codes(tree[3], prefix+'0', codes)
        get_codes(tree[4], prefix+'1', codes)
    return codes

tree = build_huffman(probs)
codes = get_codes(tree, '', {})

print('Sample Huffman codewords (10 most frequent symbols):')
for sym, p in sorted(probs.items(), key=lambda x: -x[1])[:10]:
    display = repr(sym) if sym in (' ','\n','\t') else sym
    print(f'  "{display}": P={p:.4f}, code={codes[sym]}, length={len(codes[sym])}')

Sample Huffman codewords (10 most frequent symbols):
  "' '": P=0.1628, code=110, length=3
  "e": P=0.0948, code=000, length=3
  "t": P=0.0839, code=1110, length=4
  "o": P=0.0663, code=1010, length=4
  "a": P=0.0602, code=1001, length=4
  "i": P=0.0589, code=1000, length=4
  "r": P=0.0551, code=0110, length=4
  "n": P=0.0538, code=0101, length=4
  "s": P=0.0527, code=0100, length=4
  "h": P=0.0439, code=11111, length=5


### Q1b: Encode the text and measure average bits per symbol

In [3]:
# Encode first 10000 symbols for speed
sample = symbols[:10000]
bitstream = ''.join(codes[s] for s in sample)

avg_len = len(bitstream) / len(sample)
print(f'Encoded {len(sample)} symbols -> {len(bitstream)} bits')
print(f'Average bits/symbol : {avg_len:.4f}')
print(f'H(X)                : {H:.4f}')
print(f'Gap to entropy      : {avg_len - H:.4f} bits  (should be < 1)')

Encoded 10000 symbols -> 43539 bits
Average bits/symbol : 4.3539
H(X)                : 4.3104
Gap to entropy      : 0.0435 bits  (should be < 1)


### Q1c: Compare with entropy bound H(X) <= E[l(X)] < H(X)+1

In [4]:
E_l = sum(probs[s]*len(codes[s]) for s in probs)
print(f'E[l(X)] from code table  : {E_l:.4f} bits/symbol')
print(f'H(X)                     : {H:.4f}')
print(f'H(X) + 1                 : {H+1:.4f}')
print(f'H(X) <= E[l(X)] < H(X)+1 : {H:.4f} <= {E_l:.4f} < {H+1:.4f} -> {H <= E_l < H+1}')

E[l(X)] from code table  : 4.3443 bits/symbol
H(X)                     : 4.3104
H(X) + 1                 : 5.3104
H(X) <= E[l(X)] < H(X)+1 : 4.3104 <= 4.3443 < 5.3104 -> True


### Q1d: Decode to verify lossless compression

In [5]:
def huffman_decode(bitstream, codes):
    # Reverse map: codeword -> symbol
    rev = {v: k for k,v in codes.items()}
    decoded = []
    buf = ''
    for bit in bitstream:
        buf += bit
        if buf in rev:
            decoded.append(rev[buf])
            buf = ''
    return decoded

decoded = huffman_decode(bitstream, codes)
match = (decoded == sample)
print(f'Decoded {len(decoded)} symbols')
print(f'Matches original: {match}')
if match:
    print('Lossless compression verified!')

Decoded 10000 symbols
Matches original: True
Lossless compression verified!


---
## Task 2 — Block Coding
### Q2a: Huffman on blocks of size n=2,3

In [6]:
def block_huffman_avg(sym_list, block_size):
    # Build blocks and their empirical probabilities
    blocks = [tuple(sym_list[i:i+block_size])
              for i in range(0, len(sym_list)-block_size+1, block_size)]
    n_b = len(blocks)
    block_counts = collections.Counter(blocks)
    block_probs  = {b: c/n_b for b,c in block_counts.items()}

    # Build Huffman on blocks
    tree = build_huffman(block_probs)
    codes_b = get_codes(tree, '', {})

    E_l_block = sum(block_probs[b]*len(codes_b[b]) for b in block_probs)
    return E_l_block / block_size  # per symbol

print('Block Huffman — average bits per symbol:')
results = []
for bs in [1, 2, 3]:
    avg = block_huffman_avg(symbols[:20000], bs)
    results.append((bs, avg))
    print(f'  Block size {bs}: {avg:.4f} bits/symbol')
print(f'  H(X)         : {H:.4f} bits/symbol')

Block Huffman — average bits per symbol:
  Block size 1: 4.3316 bits/symbol
  Block size 2: 3.8115 bits/symbol
  Block size 3: 3.2592 bits/symbol
  H(X)         : 4.3104 bits/symbol


### Q2b: Plot convergence toward entropy rate

In [ ]:
plt.figure(figsize=(7,4))
plt.plot([r[0] for r in results], [r[1] for r in results], marker='o', label='Block Huffman')
plt.axhline(H, color='red', linestyle='--', label=f'H(X) = {H:.4f}')
plt.xlabel('Block size')
plt.ylabel('Bits/symbol')
plt.title('Block Huffman convergence — Text (character level)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('text_block_huffman.png', dpi=100)
plt.show()
print('Larger blocks get closer to H(X) but alphabet size grows exponentially.')

---
## Task 3 — Arithmetic Coding
### Q3a: Implement arithmetic coding — memoryless model

In [ ]:
def build_cumulative(prob_dict):
    # Sort alphabet and build cumulative probability table
    alphabet = sorted(prob_dict.keys())
    cum = {}
    c = 0.0
    for sym in alphabet:
        cum[sym] = (c, c + prob_dict[sym])
        c += prob_dict[sym]
    return alphabet, cum

def arithmetic_encode(seq, prob_dict):
    alphabet, cum = build_cumulative(prob_dict)
    L, U = 0.0, 1.0
    bits = []
    pending = 0  # pending bits for underflow

    for sym in seq:
        lo, hi = cum[sym]
        width = U - L
        U = L + width * hi
        L = L + width * lo

        # Renormalization: emit bits as interval is determined
        while True:
            if U <= 0.5:          # both in [0, 0.5) -> emit 0
                bits.append(0)
                bits.extend([1]*pending); pending = 0
                L *= 2; U *= 2
            elif L >= 0.5:        # both in [0.5, 1) -> emit 1
                bits.append(1)
                bits.extend([0]*pending); pending = 0
                L = (L-0.5)*2; U = (U-0.5)*2
            elif L >= 0.25 and U <= 0.75:  # underflow
                pending += 1
                L = (L-0.25)*2; U = (U-0.25)*2
            else:
                break

    # Flush remaining bits
    pending += 1
    if L < 0.25:
        bits.append(0); bits.extend([1]*pending)
    else:
        bits.append(1); bits.extend([0]*pending)

    return bits

# Test on small sample
sample_ac = symbols[:1000]
bits_ac = arithmetic_encode(sample_ac, probs)
avg_ac = len(bits_ac) / len(sample_ac)
print(f'Arithmetic coding (memoryless, 1000 symbols):')
print(f'  Bits output     : {len(bits_ac)}')
print(f'  Bits/symbol     : {avg_ac:.4f}')
print(f'  H(X)            : {H:.4f}')
print(f'  Overhead        : {avg_ac - H:.4f} bits')

### Q3b: Arithmetic coding with 1st-order Markov model

In [ ]:
# Build transition probabilities P(next | current)
trans_counts = collections.defaultdict(collections.Counter)
for i in range(len(symbols)-1):
    trans_counts[symbols[i]][symbols[i+1]] += 1

trans_probs = {}
for ctx, cnts in trans_counts.items():
    total = sum(cnts.values())
    trans_probs[ctx] = {sym: c/total for sym,c in cnts.items()}

def arithmetic_encode_markov(seq, trans_probs, init_probs):
    bits = []
    L, U = 0.0, 1.0
    pending = 0
    prev = None

    for sym in seq:
        # Choose probability model based on previous symbol
        if prev is None or prev not in trans_probs:
            p_model = init_probs
        else:
            p_model = trans_probs[prev]

        # Build cumulative for this context
        _, cum = build_cumulative(p_model)
        if sym not in cum:
            prev = sym
            continue

        lo, hi = cum[sym]
        width = U - L
        U = L + width * hi
        L = L + width * lo

        while True:
            if U <= 0.5:
                bits.append(0); bits.extend([1]*pending); pending=0
                L*=2; U*=2
            elif L >= 0.5:
                bits.append(1); bits.extend([0]*pending); pending=0
                L=(L-0.5)*2; U=(U-0.5)*2
            elif L>=0.25 and U<=0.75:
                pending+=1; L=(L-0.25)*2; U=(U-0.25)*2
            else:
                break
        prev = sym

    pending += 1
    if L < 0.25: bits.append(0); bits.extend([1]*pending)
    else:        bits.append(1); bits.extend([0]*pending)
    return bits

bits_markov = arithmetic_encode_markov(sample_ac, trans_probs, probs)
avg_markov  = len(bits_markov) / len(sample_ac)
print(f'Arithmetic coding (1st-order Markov, 1000 symbols):')
print(f'  Bits/symbol : {avg_markov:.4f}')
print(f'  Memoryless  : {avg_ac:.4f}')
print(f'  H(X)        : {H:.4f}')
print(f'  Markov model captures memory -> better compression!')

### Q3c: Adaptive arithmetic coding with smoothed probability estimation

In [ ]:
def arithmetic_encode_adaptive(seq, alphabet, alpha=0.5):
    # Start with uniform + smoothing, update counts as we go
    adaptive_counts = {sym: alpha for sym in alphabet}
    bits = []
    L, U = 0.0, 1.0
    pending = 0

    for sym in seq:
        total = sum(adaptive_counts.values())
        p_model = {s: c/total for s,c in adaptive_counts.items()}
        _, cum = build_cumulative(p_model)

        if sym not in cum: continue
        lo, hi = cum[sym]
        width = U - L
        U = L + width * hi
        L = L + width * lo

        while True:
            if U <= 0.5:
                bits.append(0); bits.extend([1]*pending); pending=0
                L*=2; U*=2
            elif L >= 0.5:
                bits.append(1); bits.extend([0]*pending); pending=0
                L=(L-0.5)*2; U=(U-0.5)*2
            elif L>=0.25 and U<=0.75:
                pending+=1; L=(L-0.25)*2; U=(U-0.25)*2
            else:
                break

        adaptive_counts[sym] += 1  # update model

    pending += 1
    if L < 0.25: bits.append(0); bits.extend([1]*pending)
    else:        bits.append(1); bits.extend([0]*pending)
    return bits

alphabet = sorted(probs.keys())
bits_adaptive = arithmetic_encode_adaptive(sample_ac, alphabet)
avg_adaptive  = len(bits_adaptive) / len(sample_ac)
print(f'Adaptive arithmetic coding (1000 symbols):')
print(f'  Bits/symbol : {avg_adaptive:.4f}')
print(f'  Memoryless  : {avg_ac:.4f}')
print(f'  Adaptive starts with uniform prior and learns — converges as n grows.')

### Q3d: Effect of block length on arithmetic coding rate

In [ ]:
block_sizes = [100, 500, 1000, 5000]
ac_rates = []
for bs in block_sizes:
    b = arithmetic_encode(symbols[:bs], probs)
    ac_rates.append(len(b)/bs)

plt.figure(figsize=(7,4))
plt.semilogx(block_sizes, ac_rates, marker='o', label='Arithmetic coding')
plt.axhline(H, color='red', linestyle='--', label=f'H(X) = {H:.4f}')
plt.xlabel('Block length (log scale)')
plt.ylabel('Bits/symbol')
plt.title('Arithmetic coding rate vs block length — Text')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('text_arith_vs_blocklen.png', dpi=100)
plt.show()

---
## Task 4 — Analysis and Comparison
### Q4a: Summary table — all methods

In [ ]:
print('=== Compression Results — Text (character level) ===')
print(f'{"Method":<35} {"Bits/symbol":>12}')
print('-'*50)
print(f'{"H(X) — entropy bound":<35} {H:>12.4f}')
print(f'{"Huffman (symbol)":<35} {E_l:>12.4f}')
for bs, avg in results:
    print(f'{f"Block Huffman (n={bs})":<35} {avg:>12.4f}')
print(f'{"Arithmetic (memoryless)":<35} {avg_ac:>12.4f}')
print(f'{"Arithmetic (Markov k=1)":<35} {avg_markov:>12.4f}')
print(f'{"Arithmetic (adaptive)":<35} {avg_adaptive:>12.4f}')

### Q4b: Compare with system compressors (gzip, bzip2, xz)

In [ ]:
import os, subprocess, tempfile

# Write text to a temp file
with tempfile.NamedTemporaryFile(delete=False, suffix='.txt', mode='w') as f:
    f.write(raw)
    tmp_path = f.name

original_bytes = os.path.getsize(tmp_path)
original_bits_per_char = 8  # uncompressed = 8 bits per byte = 8 bits per char (ASCII)

for tool, flag in [('gzip', '-k'), ('bzip2', '-k'), ('xz', '-k')]:
    out = tmp_path + ('gz' if tool=='gzip' else 'bz2' if tool=='bzip2' else 'xz')
    try:
        subprocess.run([tool, flag, tmp_path], capture_output=True)
        ext = '.gz' if tool=='gzip' else '.bz2' if tool=='bzip2' else '.xz'
        out = tmp_path + ext
        if os.path.exists(out):
            comp_bytes = os.path.getsize(out)
            ratio = comp_bytes / original_bytes
            bps = ratio * 8
            print(f'{tool:8s}: {comp_bytes} bytes, {bps:.4f} bits/char, ratio={ratio:.4f}')
            os.remove(out)
    except Exception as e:
        print(f'{tool}: not available ({e})')

os.remove(tmp_path)
print(f'\nFor reference: H(X) = {H:.4f} bits/char')